In [2]:
import random
from tqdm import tqdm

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:115.0) Gecko/20100101 Firefox/115.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:115.0) Gecko/20100101 Firefox/115.0",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.1.2 Safari/605.1.15",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Edg/115.0.0.0",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 OPR/85.0.4341.72",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 OPR/85.0.4341.72",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Vivaldi/5.3.2679.55",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Vivaldi/5.3.2679.55",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Brave/1.40.107",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 Brave/1.40.107",
]

random.choice(USER_AGENTS)

'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36 OPR/85.0.4341.72'

In [ ]:
import time
from tqdm import tqdm
import requests
import pandas as pd
from bs4 import BeautifulSoup
import re

class Results:
    @staticmethod
    def scrape(race_id_list):
        """
        レース結果データをスクレイピングする関数
        Parameters:
        ----------
        race_id_list : list
            レースIDのリスト
        Returns:
        ----------
        race_results_df : pandas.DataFrame
            全レース結果データをまとめてDataFrame型にしたもの
        """
        race_results = {}
        for race_id in tqdm(race_id_list):
            print(f"アクセス開始: {race_id}")
            time.sleep(2)
            try:
                url = "https://db.netkeiba.com/race/" + str(race_id)
                headers = {'User-Agent': random.choice(USER_AGENTS)}
                html = requests.get(url, headers=headers)
                html.raise_for_status()
                html.encoding = "EUC-JP"
                
                soup = BeautifulSoup(html.text, "html.parser")
                
                # レース結果テーブルを直接解析
                try:
                    race_table = soup.find("table", attrs={"summary": "レース結果"})
                    if not race_table:
                        continue
                    
                    # ヘッダー行を取得
                    header_row = race_table.find("tr")
                    headers = []
                    if header_row:
                        for th in header_row.find_all("th"):
                            headers.append(th.text.strip())
                    
                    # データ行を取得
                    data_rows = []
                    for tr in race_table.find_all("tr")[1:]:  # ヘッダー行をスキップ
                        row_data = []
                        for td in tr.find_all("td"):
                            row_data.append(td.text.strip())
                        if row_data:  # 空行でない場合のみ追加
                            data_rows.append(row_data)
                    
                    print(f"データ行数: {len(data_rows)}")
                    if data_rows:
                        print(f"最初のデータ行: {data_rows[0]}")
                    
                    # DataFrameを作成
                    if data_rows and headers:
                        # ヘッダーとデータの長さを合わせる
                        max_cols = max(len(headers), max(len(row) for row in data_rows))
                        
                        # ヘッダーを調整
                        while len(headers) < max_cols:
                            headers.append(f"col_{len(headers)}")
                        
                        # データ行を調整
                        for row in data_rows:
                            while len(row) < max_cols:
                                row.append("")
                        
                        df = pd.DataFrame(data_rows, columns=headers[:max_cols])
                        print(f"作成したDataFrameの列: {list(df.columns)}")
                        
                        # 除外する列のキーワード
                        exclude_keywords = [
                            "着差", "ﾀｲﾑ指数", "調教ﾀｲﾑ", "厩舎ｺﾒﾝﾄ", "備考", "馬主", "賞金"
                        ]
                        
                        # 除外する列を特定
                        columns_to_drop = []
                        for col in df.columns:
                            for keyword in exclude_keywords:
                                if keyword in col:
                                    columns_to_drop.append(col)
                                    break
                        
                        # 列を除外
                        df = df.drop(columns=columns_to_drop, errors='ignore')
                        
  
                    else:
                        print("データまたはヘッダーが取得できませんでした")
                        continue
                        
                except Exception as e:
                    print(f"テーブル解析失敗: {e}")
                    # フォールバック: pandas.read_htmlを使用
                    try:
                        tables = pd.read_html(html.text)
                        print(f"フォールバック: テーブル数: {len(tables)}")
                        df = tables[0]
                        print(f"フォールバック列名: {list(df.columns)}")
                    except Exception as e2:
                        print(f"フォールバックも失敗: {e2}")
                        continue
                
                df = df.rename(columns=lambda x: x.replace(' ', ''))
                
                # 天候、レースの種類、コースの長さ、馬場の状態、日付をスクレイピング
                try:
                    data_intro = soup.find("div", attrs={"class": "data_intro"})
                    if data_intro:
                        ps = data_intro.find_all("p")
                        texts = ps[0].text + ps[1].text
                    else:
                        print("data_introが見つかりません")
                        continue
                except Exception as e:
                    print(f"BeautifulSoup取得失敗: {e}")
                    continue
                info = re.findall(r'\w+', texts)
                for text in info:
                    if text in ["芝", "ダート"]:
                        df["race_type"] = [text] * len(df)
                    if "障" in text:
                        df["race_type"] = ["障害"] * len(df)
                    if "m" in text:
                        df["course_len"] = [int(re.findall(r"\d+", text)[-1])] * len(df)
                    if text in ["良", "稍重", "重", "不良"]:
                        df["ground_state"] = [text] * len(df)
                    if text in ["曇", "晴", "雨", "小雨", "小雪", "雪"]:
                        df["weather"] = [text] * len(df)
                    if "年" in text:
                        df["date"] = [text] * len(df)
                        
                #馬ID、騎手IDをスクレイピング
                try:
                    horse_id_list = []
                    horse_a_list = soup.find("table", attrs={"summary": "レース結果"}).find_all(
                        "a", attrs={"href": re.compile("^/horse")}
                    )
                    for a in horse_a_list:
                        horse_id = re.findall(r"\d+", a["href"])
                        horse_id_list.append(horse_id[0])
                    jockey_id_list = []
                    jockey_a_list = soup.find("table", attrs={"summary": "レース結果"}).find_all(
                        "a", attrs={"href": re.compile("^/jockey")}
                    )
                    for a in jockey_a_list:
                        jockey_id = re.findall(r"\d+", a["href"])
                        jockey_id_list.append(jockey_id[0])
                    df["horse_id"] = horse_id_list
                    df["jockey_id"] = jockey_id_list
                except Exception as e:
                    print(f"ID取得失敗: {e}")
                    continue
                    
                df.index = [race_id] * len(df)
                race_results[race_id] = df
                print(f"データ取得成功: {len(df)}頭")
                
            except IndexError:
                print("IndexError: 存在しないrace_id")
                continue
            except AttributeError:
                print("AttributeError: 存在しないrace_id")
                continue
            except Exception as e:
                print(f"その他の例外: {e}")
                break
            except:
                print("その他の例外（詳細不明）")
                break
        if race_results:
            race_results_df = pd.concat([race_results[key] for key in race_results])
        else:
            race_results_df = pd.DataFrame()
        return race_results_df

In [ ]:
import pickle
import os

def generate_race_id_list(year):
    """
    指定した年のレースIDリストを生成する関数
    Parameters:
    ----------
    year : int
        対象年
    Returns:
    ----------
    race_id_list : list
        レースIDのリスト
    """
    race_id_list = []
    for place in range(1, 11, 1):
        for kai in range(1, 7, 1):
            for day in range(1, 13, 1):
                for r in range(1, 13, 1):
                    race_id = str(year).zfill(2) + str(place).zfill(2) + str(kai).zfill(2) + str(day).zfill(2) + str(r).zfill(2)
                    race_id_list.append(race_id)
    return race_id_list

def scrape_and_save_race_data(year):
    """
    指定した年のレースデータをスクレイピングしてpickleファイルに保存する関数
    Parameters:
    ----------
    year : int
        対象年
    """
    # レースIDリストを生成
    race_id_list = generate_race_id_list(year)
    print(f"{year}年のレースIDリスト生成完了: {len(race_id_list)}件")
    
    # スクレイピング実行
    results = Results.scrape(race_id_list)
    
    # pickleファイルに保存
    os.makedirs('data', exist_ok=True)
    filename = f"race_results_{year}.pkl"
    with open(filename, 'wb') as f:
        pickle.dump(results, f)
    
    print(f"{year}年のデータを{filename}に保存しました")
    return results

# 2019年のデータをスクレイピングして保存
results_2019 = scrape_and_save_race_data(2019)



2019年のレースIDリスト生成完了: 8640件


  0%|          | 0/8640 [00:00<?, ?it/s]

アクセス開始: 201901010101


  0%|          | 1/8640 [00:01<2:47:29,  1.16s/it]

テーブルヘッダー: ['着順', '枠番', '馬番', '馬名', '性齢', '斤量', '騎手', 'タイム', '着差', 'ﾀｲﾑ指数', '通過', '上り', '単勝', '人気', '馬体重', '調教ﾀｲﾑ', '厩舎ｺﾒﾝﾄ', '備考', '調教師', '馬主', '賞金(万円)']
データ行数: 9
最初のデータ行: ['1', '1', '1', 'ゴルコンダ', '牡2', '54', 'ルメール', '1:48.3', '', '**', '1-1-1-1', '36.5', '1.4', '1', '518(-16)', '', '', '', '[東]\n木村哲也', 'サンデーレーシング', '500.0']
作成したDataFrameの列: ['着順', '枠番', '馬番', '馬名', '性齢', '斤量', '騎手', 'タイム', '着差', 'ﾀｲﾑ指数', '通過', '上り', '単勝', '人気', '馬体重', '調教ﾀｲﾑ', '厩舎ｺﾒﾝﾄ', '備考', '調教師', '馬主', '賞金(万円)']
データ取得成功: 9頭
アクセス開始: 201901010102


...

テーブルヘッダー: ['着順', '枠番', '馬番', '馬名', '性齢', '斤量', '騎手', 'タイム', '着差', 'ﾀｲﾑ指数', '通過', '上り', '単勝', '人気', '馬体重', '調教ﾀｲﾑ', '厩舎ｺﾒﾝﾄ', '備考', '調教師', '馬主', '賞金(万円)']
データ行数: 12
最初のデータ行: ['1', '2', '2', 'マーマレードガール', '牝3', '52', '丹内祐次', '1:08.5', '', '**', '1-1', '34.8', '4.3', '3', '456(-4)', '', '', '', '[東]\n中野栄治', 'ビッグレッドファーム', '750.0']
作成したDataFrameの列: ['着順', '枠番', '馬番', '馬名', '性齢', '斤量', '騎手', 'タイム', '着差', 'ﾀｲﾑ指数', '通過', '上り', '単勝', '人気', '馬体重', '調教ﾀｲﾑ', '厩舎ｺﾒﾝﾄ', '備考', '調教師', '馬主', '賞金(万円)']
データ取得成功: 12頭
アクセス開始: 201901010109


  0%|          | 9/8640 [00:12<3:23:39,  1.42s/it]

テーブルヘッダー: ['着順', '枠番', '馬番', '馬名', '性齢', '斤量', '騎手', 'タイム', '着差', 'ﾀｲﾑ指数', '通過', '上り', '単勝', '人気', '馬体重', '調教ﾀｲﾑ', '厩舎ｺﾒﾝﾄ', '備考', '調教師', '馬主', '賞金(万円)']
データ行数: 10
最初のデータ行: ['1', '3', '3', 'ヨハン', '牡3', '54', '古川吉洋', '1:45.1', '', '**', '3-3-3-4', '36.5', '1.7', '1', '432(+2)', '', '', '', '[西]\n高橋康之', 'サイプレスホールディングス', '750.0']
作成したDataFrameの列: ['着順', '枠番', '馬番', '馬名', '性齢', '斤量', '騎手', 'タイム', '着差', 'ﾀｲﾑ指数', '通過', '上り', '単勝', '人気', '馬体重', '調教ﾀｲﾑ', '厩舎ｺﾒﾝﾄ', '備考', '調教師', '馬主', '賞金(万円)']
データ取得成功: 10頭
アクセス開始: 201901010110


  0%|          | 9/8640 [00:13<3:33:05,  1.48s/it]


KeyboardInterrupt: 